# 🤖 ChatMateAI Chatbot

This project is a simple Generative AI chatbot built using a pre-trained language model from Hugging Face. It takes user input, processes it using a transformer-based model, and generates human-like responses in real time.

The chatbot is deployed using Gradio, providing an interactive web interface where users can chat seamlessly. It runs on GPU (if available) for faster response generation.

## 🚀 Features
- Text-based conversational AI
- Uses transformer model for response generation
- GPU acceleration support (CUDA)
- Interactive UI using Gradio
- Simple and clean implementation for learning GenAI basics

## 🛠️ Tech Stack
- Transformers (Hugging Face)
- PyTorch
- LangChain (for memory concept)
- Gradio (UI)

## 📌 Use Case
- Learning Generative AI basics
- Building chatbot applications
- Understanding LLM inference pipeline

### Installing Required Libraries

- transformers → for loading pre-trained LLMs (Hugging Face)
- langchain → for building chatbot logic and chaining
- torch → deep learning backend (model execution)
- gradio → to create UI for chatbot

Command:
pip install transformers langchain torch gradio

In [1]:
!pip install transformers langchain torch gradio

In [2]:
!pip uninstall -y langchain
!pip install langchain==0.1.0

Found existing installation: langchain 1.2.12
Uninstalling langchain-1.2.12:
  Successfully uninstalled langchain-1.2.12
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which ver

In [3]:
!pip install huggingface_hub

from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain.memory.buffer import ConversationBufferMemory
import gradio as gr

In [3]:
# Load model and tokenizer
model_name = "google/gemma-2-2b-it"
token = "YOUR_SECRET_TOKEN"

# Check if CUDA is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer with token
model = AutoModelForCausalLM.from_pretrained(model_name, token=token).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)

Using device: cuda


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [4]:
# Function to generate chatbot response without chat history
def chatbot_response(user_input):
    # Tokenize input and move to GPU
    inputs = tokenizer(user_input, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate response
    outputs = model.generate(**inputs)

    # Decode output
    bot_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Ensure the bot doesn't repeat the user input
    if bot_reply.startswith(user_input):
        bot_reply = bot_reply[len(user_input):].strip()

    return bot_reply

In [5]:
chatbot_response("Hi how are you?")

"I'm doing well, thank you. How are you?\n\nI'm great,"

In [7]:
def chatbot_interface(user_input, history):
    response = chatbot_response(user_input)
    history.append((user_input, response))
    return history, ""

# Creating the Gradio Interface
demo = gr.Blocks()

with demo:
    gr.Markdown("## ChatMateAI Chatbot")
    chatbot = gr.Chatbot()
    with gr.Row():
        user_input = gr.Textbox(placeholder="Type your message here...", lines=1)
        send_button = gr.Button("Send")

    history = gr.State([])

    send_button.click(chatbot_interface, inputs=[user_input, history], outputs=[chatbot, user_input])
    user_input.submit(chatbot_interface, inputs=[user_input, history], outputs=[chatbot, user_input])

# Launch the chatbot
demo.launch(share = True)

/tmp/ipykernel_2210/3075273882.py:11: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_2210/3075273882.py:11: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2df43d2d00172383d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
